#### INST326 OOP Project 04

Rename this notebook, replacing "_Assignment" with "_YourName"<br>
Insert Signature Block Here

In [7]:
### Your Name: Amr Khaled

> INST326 Section 0202
>  
> Group number / name if applicable  
> Assignment: Project 4
>  
> Date Due:12/11/24 11:59
>  
> PIN: 
#### Honor Pledge

> I pledge that the work contained in this assignment is my own...
> 
#### Disclosures

> None.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3482739943.py, line 3)

You may work as an individual on **ONE** of the following projects, **OR** if you want to work as a group, contact Dr. Dempwolf for a project assignment. That group assignment will be part of an ongoing research project analyzing innovation ecosystems. 

### Individual Projects
Choose **ONE** of the following projects and write to code solution in the code cell below your choice. Use comments in your code to document your solution. If you need to write comments to the grader, add a markdown cell immediately above your code solution and add your comments there. Be sure to read and follow the Notebook Instructions at the bottom of this notebook. Your grade may depend on it! 

#### 1. Library Management System>  Objective: Develop a system to manage a library’s collection of books, users, and loan records. This system should allow users to borrow and return books, as well as track which books are currently available.
>
> Requirements
>>- Use classes to represent books, users, and the library.
>>- Implement encapsulation to protect class attributes.
>>- Use inheritance to handle different types of users (e.g., students and teachers).
>>- Demonstrate polymorphism in borrowing rules (e.g., different borrowing limits for students vs. teachers).
>>- Include methods for adding/removing books, registering users, and managing book loans.
>>- Include execution code to demonstrate that your solution works

In [11]:
# Solution - enter your code solution below
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from typing import List, Optional, Union

class LibraryError(Exception):
    pass  # Custom exception for library-related errors.

class Person(ABC):
    def __init__(self, name: str, email: str):
        if not name or not email:
            raise ValueError("Name and email cannot be empty")  # Validate inputs.
        self._name: str = name
        self._email: str = email
        self._borrowed_books: List[Book] = []  # List to track borrowed books.

    @property
    def name(self) -> str:
        return self._name  # Getter for name.

    @property
    def email(self) -> str:
        return self._email  # Getter for email.

    @property
    def borrowed_books(self) -> List['Book']:
        return self._borrowed_books.copy()  # Getter for borrowed books (copy for safety).

    @abstractmethod
    def can_borrow(self, num_books: int = 1) -> bool:
        pass  # Determine if the user can borrow books.

class Student(Person):
    def __init__(self, name: str, email: str, student_id: str):
        super().__init__(name, email)
        self._student_id: str = student_id  # Unique student ID.
        self._max_books: int = 3  # Borrowing limit for students.

    @property
    def student_id(self) -> str:
        return self._student_id  # Getter for student ID.

    def can_borrow(self, num_books: int = 1) -> bool:
        return len(self._borrowed_books) + num_books <= self._max_books  # Check limit.

class Teacher(Person):
    def __init__(self, name: str, email: str, department: str):
        super().__init__(name, email)
        self._department: str = department  # Teacher's department.
        self._max_books: int = 5  # Borrowing limit for teachers.

    @property
    def department(self) -> str:
        return self._department  # Getter for department.

    def can_borrow(self, num_books: int = 1) -> bool:
        return len(self._borrowed_books) + num_books <= self._max_books  # Check limit.

class Book:
    def __init__(self, title: str, author: str, isbn: str):
        if not all([title, author, isbn]):
            raise ValueError("Title, author, and ISBN cannot be empty")  # Validate inputs.
        self._title: str = title
        self._author: str = author
        self._isbn: str = isbn
        self._is_available: bool = True  # Book starts as available.
        self._loan_history: List[dict] = []  # Record of loans.

    @property
    def title(self) -> str:
        return self._title  # Getter for title.

    @property
    def author(self) -> str:
        return self._author  # Getter for author.

    @property
    def isbn(self) -> str:
        return self._isbn  # Getter for ISBN.

    @property
    def is_available(self) -> bool:
        return self._is_available  # Check if book is available.

    def update_availability(self, status: bool) -> None:
        self._is_available = status  # Update availability.

    def add_loan_record(self, user: Person, loan_date: datetime) -> None:
        self._loan_history.append({
            'user': user.name,
            'loan_date': loan_date,
            'return_date': None
        })  # Record loan details.

class Library:
    def __init__(self):
        self._books: List[Book] = []  # List of books in the library.
        self._users: List[Person] = []  # List of registered users.
        self._active_loans: dict = {}  # Track active loans.

    def add_book(self, book: Book) -> None:
        if any(existing.isbn == book.isbn for existing in self._books):
            raise LibraryError(f"Book with ISBN {book.isbn} already exists")  # Check duplicates.
        self._books.append(book)  # Add book to library.

    def register_user(self, user: Person) -> None:
        if any(existing.email == user.email for existing in self._users):
            raise LibraryError(f"User with email {user.email} already registered")  # Check duplicates.
        self._users.append(user)  # Add user to library.

    def find_book(self, title: str) -> Optional[Book]:
        return next((book for book in self._books if book.title.lower() == title.lower()), None)  # Search by title.

    def borrow_book(self, user: Person, book: Book) -> bool:
        if not book.is_available:
            raise LibraryError(f"{book.title} is not available")  # Check availability.
        if not user.can_borrow():
            raise LibraryError(f"{user.name} has reached maximum book limit")  # Check limit.
        book.update_availability(False)  # Mark book as unavailable.
        user._borrowed_books.append(book)  # Add book to user's borrowed list.
        loan_date = datetime.now()  # Record loan date.
        book.add_loan_record(user, loan_date)  # Add to loan history.
        self._active_loans[book] = {
            'user': user,
            'borrowed_date': loan_date,
            'due_date': loan_date + timedelta(days=14)  # Calculate due date.
        }
        return True  # Borrow successful.

    def return_book(self, user: Person, book: Book) -> bool:
        if book not in user.borrowed_books:
            raise LibraryError("Book was not borrowed by this user")  # Validate return.
        book.update_availability(True)  # Mark book as available.
        user._borrowed_books.remove(book)  # Remove from user's list.
        if book in self._active_loans:
            del self._active_loans[book]  # Remove from active loans.
        return True  # Return successful.

def main():
    try:
        library = Library()  # Initialize library.
        book1 = Book("Python Programming", "Layla", "123456")
        book2 = Book("OOP Principles", "Menna", "789012")
        library.add_book(book1)  # Add books.
        library.add_book(book2)

        student = Student("Amr", "akhaled@example.com", "S12345")
        teacher = Teacher("Will", "Will@example.com", "Computer Science")
        library.register_user(student)  # Register student.
        library.register_user(teacher)  # Register teacher.

        library.borrow_book(student, book1)  # Student borrows book.
        library.borrow_book(teacher, book2)  # Teacher borrows book.

        library.return_book(student, book1)  # Student returns book.

        print("Library management system demonstration completed successfully")  # Success message.

    except LibraryError as e:
        print(f"Library Error: {e}")  # Handle library-specific errors.
    except Exception as e:
        print(f"Unexpected error: {e}")  # Handle unexpected errors.

if __name__ == "__main__":
    main()  # Run the demonstration.


Library management system demonstration completed successfully


#### 2. Online Shopping Cart System
>  Objective: Build a shopping cart system for an online store that manages products, shopping carts, and orders.
>
> Requirements
>>- Use classes to represent products, shopping carts, and orders.
>>- Implement encapsulation to handle product prices and cart contents securely.
>>- Use inheritance to create different types of products (e.g., electronics, clothing).
>>- Demonstrate polymorphism by calculating discounts based on product type.
>>- Include execution code to demonstrate that your solution works

In [ ]:
# Solution - enter your code solution below


#### 3. Restaurant Reservation System
>  Objective: Create a reservation system for a restaurant that manages tables, reservations, and customers.
>
>  Requirements
>>- Use classes to represent tables, customers, and reservations.
>>- Implement encapsulation for managing table availability and reservation details.
>>- Use inheritance to differentiate between walk-in and advance reservations.
>>- Demonstrate polymorphism by handling special cases (e.g., priority seating for VIP customers).
>>- Include execution code to demonstrate that your solution works

In [ ]:
# Solution - enter your code solution below


#### 4. Vehicle Rental System
>  Objective: Develop a vehicle rental system that manages a fleet of vehicles, customer rentals, and payment processing.
>
>  Requirements
>>- Use classes to represent different types of vehicles, customers, and rental transactions.
>>- Implement encapsulation to handle sensitive information like customer payment details.
>>- Use inheritance to differentiate between various vehicle types (e.g., cars, trucks, motorcycles).
>>- Demonstrate polymorphism by applying different rental pricing strategies based on vehicle type.
>>- Include execution code to demonstrate that your solution works

In [ ]:
# Solution - enter your code solution below


#### 5. Online Learning Platform
>  Objective: Create an online learning platform that manages courses, students, and instructors.
>  
>  Requirements
>>- Use classes to represent courses, students, and instructors.
>>- Implement encapsulation to manage sensitive information like student grades.
>>- Use inheritance to handle different types of courses (e.g., free, paid, and premium).
>>- Demonstrate polymorphism in applying different grading schemes for assignments.
>>- Include execution code to demonstrate that your solution works

In [ ]:
# Solution - enter your code solution below


#### 6. E-Commerce Order Processing System
>  Objective: Build an order processing system for an online store that manages products, customers, and orders.
>  
>  Requirements
>>- Use classes to represent products, customers, and orders.
>>- Implement encapsulation for handling payment details securely.
>>- Use inheritance for different types of products (e.g., physical goods, digital downloads).
>>- Demonstrate polymorphism by applying different shipping costs based on product type.
>>- Include execution code to demonstrate that your solution works

In [ ]:
# Solution - enter your code solution below


### Notebook Instructions
> Before turning in your notebook:
> 1. Make sure you have renamed the notebook file as instructed
> 2. Make sure you have included your signature block and that it is correct according to the instructions
> 3. comment your code as necessary
> 4. run all code cells and double check that they run correctly. Include you execution code in your submission. If you can't get your code to run correctly and you want partial credit, add a note for the grader in a new markdown cell directly above your code solution.<br><br>
Turn in your notebook by uploading it to ELMS<br>
IF the exercises involve saved data files, put your notebook and the data file(s) in a zip folder and upload the zip folder to ELMS